In [1]:
from pathlib import Path
import os, sys

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

%load_ext autoreload
%autoreload 2

print("Working directory:", Path.cwd())

Working directory: C:\Users\makar\Python\auto-data-scientist


In [2]:
%%writefile src/orchestrator.py
from dataclasses import dataclass, field
from pathlib import Path

from src.agents.coder import solve_step
from src.agents.planner import make_plan
from src.agents.reviewer import review_step
from src.profiler import quick_profile


@dataclass
class StepLog:
    """A record of what happened on one plan step, for the final report."""
    step_id: int
    title: str
    task: str
    coder_success: bool
    coder_attempts: int
    approved: bool
    review_reason: str
    leakage_flags: list
    code: str
    stdout: str


@dataclass
class RunResult:
    """Everything about one full pipeline run, from goal to finish (or failure)."""
    success: bool
    plan: object                 # the Plan object from the planner
    logs: list = field(default_factory=list)
    failed_at_step: int = None   # None if the whole plan completed

Writing src/orchestrator.py


In [3]:
%%writefile -a src/orchestrator.py


def _files_in(workspace: Path) -> list:
    """List files currently in the workspace, so we can tell the coder what already exists."""
    return sorted(p.name for p in workspace.iterdir() if p.is_file())


def run_pipeline(csv_path, goal: str, workspace: Path,
                 max_step_retries: int = 2, coder_attempts_per_try: int = 4,
                 verbose: bool = True) -> RunResult:
    """Run the full agent pipeline: plan the goal, then execute and review each step.

    csv_path: the dataset to work on (will be copied into the workspace as data.csv)
    goal: the plain-English objective, e.g. "predict customer churn"
    workspace: a folder the sandbox will use; created if it doesn't exist
    """
    workspace = Path(workspace)
    workspace.mkdir(parents=True, exist_ok=True)

    # Put the dataset where the sandbox expects it
    import shutil
    shutil.copy(csv_path, workspace / "data.csv")

    profile = quick_profile(workspace / "data.csv")

    if verbose:
        print("[orchestrator] Planning...")
    plan = make_plan(goal, profile)
    if verbose:
        print(f"[orchestrator] Plan has {len(plan.steps)} steps, problem_type={plan.problem_type}")

    logs = []

    for step in plan.steps:
        if verbose:
            print(f"\n[orchestrator] --- Step {step.id}: {step.title} ---")

        # Tell the coder what files already exist from previous steps,
        # so it builds on prior work instead of assuming a blank slate.
        existing_files = [f for f in _files_in(workspace) if f != "step.py"]
        task_with_context = (
            f"{step.task}\n\n"
            f"Files already present in the working directory from earlier steps "
            f"(read/reuse them as needed, e.g. via pd.read_csv or joblib.load): {existing_files}"
        )

        review = None
        outcome = None

        for retry in range(max_step_retries + 1):
            outcome = solve_step(task_with_context, profile, workspace,
                                max_attempts=coder_attempts_per_try, verbose=verbose)

            if not outcome.success:
                # The coder itself never got working code; no point reviewing broken code.
                break

            review = review_step(step.task, outcome.code, outcome.result.stdout)
            if review.approved:
                break

            if verbose:
                print(f"[orchestrator] Reviewer rejected (retry {retry + 1}/{max_step_retries}): "
                      f"{review.reason}")
            # Feed the reviewer's specific objection back in as extra task context
            task_with_context = (
                f"{step.task}\n\n"
                f"Your previous attempt was rejected by a reviewer for this reason: {review.reason}\n"
                f"Leakage/quality flags: {review.leakage_flags}\n"
                f"Write a corrected version that avoids this problem.\n\n"
                f"Files already present in the working directory: {existing_files}"
            )

        log = StepLog(
            step_id=step.id, title=step.title, task=step.task,
            coder_success=outcome.success, coder_attempts=outcome.attempts,
            approved=bool(review and review.approved),
            review_reason=review.reason if review else "Coder never produced working code.",
            leakage_flags=review.leakage_flags if review else [],
            code=outcome.code, stdout=outcome.result.stdout,
        )
        logs.append(log)

        if not log.approved:
            if verbose:
                print(f"[orchestrator] Step {step.id} FAILED. Stopping pipeline.")
            return RunResult(success=False, plan=plan, logs=logs, failed_at_step=step.id)

    if verbose:
        print("\n[orchestrator] All steps completed and approved.")
    return RunResult(success=True, plan=plan, logs=logs)

Appending to src/orchestrator.py


In [4]:
from sklearn.datasets import load_breast_cancer
from src.config import RUNS_DIR

ws = RUNS_DIR / "part4_full_run" / "workspace"
ws.mkdir(parents=True, exist_ok=True)

csv_path = ws.parent / "source_data.csv"   # kept outside the workspace, orchestrator copies it in
load_breast_cancer(as_frame=True).frame.to_csv(csv_path, index=False)

In [5]:
from src.orchestrator import run_pipeline

result = run_pipeline(
    csv_path=csv_path,
    goal="Predict whether a tumor is malignant or benign, using the 'target' column.",
    workspace=ws,
)

print("\n\nOverall success:", result.success)
if not result.success:
    print("Failed at step:", result.failed_at_step)

[orchestrator] Planning...
[orchestrator] Plan has 6 steps, problem_type=classification

[orchestrator] --- Step 1: Data Exploration ---
[coder] attempt 1/4: OK

[orchestrator] --- Step 2: Missing Values & Encoding ---
[coder] attempt 1/4: OK

[orchestrator] --- Step 3: Feature Engineering/Selection ---
[coder] attempt 1/4: OK
[orchestrator] Reviewer rejected (retry 1/2): The pipeline fits the scaler, polynomial transformer, variance filter, and SelectKBest on the entire dataset before any train‑test split, causing data leakage.
[coder] attempt 1/4: OK

[orchestrator] --- Step 4: Model Training & Comparison ---
[coder] attempt 1/4: OK

[orchestrator] --- Step 5: Best Model Evaluation & Charting ---
[coder] attempt 1/4: FAILED
[coder] attempt 2/4: FAILED
[coder] attempt 3/4: FAILED
[coder] attempt 4/4: FAILED
[orchestrator] Step 5 FAILED. Stopping pipeline.


Overall success: False
Failed at step: 5


In [6]:
for log in result.logs:
    print(f"Step {log.step_id} ({log.title}): approved={log.approved}, "
          f"coder_attempts={log.coder_attempts}, reason={log.review_reason}")

Step 1 (Data Exploration): approved=True, coder_attempts=1, reason=The code correctly loads the data, prints summary statistics and class distribution, and saves histograms and boxplots for each feature as required.
Step 2 (Missing Values & Encoding): approved=True, coder_attempts=1, reason=The code correctly checks and imputes missing values, one‑hot encodes any categorical columns, and casts the target to integer as required, with no evidence of data leakage.
Step 3 (Feature Engineering/Selection): approved=True, coder_attempts=1, reason=The pipeline correctly avoids data leakage by fitting all preprocessing steps on the training set only, and the output meets the task requirements.
Step 4 (Model Training & Comparison): approved=True, coder_attempts=1, reason=The code correctly splits the data, avoids leakage, trains two models, and reports cross‑validated accuracy and ROC‑AUC as required.
Step 5 (Best Model Evaluation & Charting): approved=False, coder_attempts=4, reason=Coder never

In [7]:
import os
print(sorted(os.listdir(ws)))
# Expect data.csv, plus whatever charts/model files the steps created (e.g. a .png and a .joblib)

['boxplots', 'data.csv', 'histograms', 'logisticregression_model.joblib', 'poly_features.joblib', 'processed_data.csv', 'processed_test.csv', 'processed_train.csv', 'randomforest_model.joblib', 'scaler.joblib', 'select_kbest.joblib', 'step.py', 'var_thresh.joblib']


In [8]:
from src.agents.coder import solve_step
from src.config import RUNS_DIR
from src.profiler import quick_profile

ws = RUNS_DIR / "part4_full_run" / "workspace"
profile = quick_profile(ws / "data.csv")

# Same task text the planner produced for step 5 — check result.plan.steps to confirm exact wording
step5_task = result.plan.steps[4].task   # index 4 = step id 5

existing_files = [f for f in __import__("os").listdir(ws) if f != "step.py"]
task_with_context = f"{step5_task}\n\nFiles already present: {existing_files}"

outcome = solve_step(task_with_context, profile, ws, max_attempts=1, verbose=True)

print("Success:", outcome.success)
print("\n--- STDOUT ---\n", outcome.result.stdout)
print("\n--- STDERR ---\n", outcome.result.stderr)
print("\n--- CODE IT WROTE ---\n", outcome.code)

[coder] attempt 1/1: FAILED
Success: False

--- STDOUT ---
 

--- STDERR ---
 Traceback (most recent call last):
  File "/usr/local/lib/python3.11/site-packages/pandas/core/indexes/base.py", line 3641, in get_loc
    return self._engine.get_loc(casted_key)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "pandas/_libs/index.pyx", line 168, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/index.pyx", line 197, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/hashtable_class_helper.pxi", line 7668, in pandas._libs.hashtable.PyObjectHashTable.get_item
  File "pandas/_libs/hashtable_class_helper.pxi", line 7676, in pandas._libs.hashtable.PyObjectHashTable.get_item
KeyError: 'target'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/workspace/step.py", line 11, in <module>
    y_val = val_df["target"]
            ~~~~~~^^^^^^^^^^
  File "/usr/local/lib/python3.11/site-packages/pandas/core/frame.py

In [9]:
import pandas as pd
from src.config import RUNS_DIR

ws = RUNS_DIR / "part4_full_run" / "workspace"
test_df = pd.read_csv(ws / "processed_test.csv")
print(test_df.columns.tolist())

['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19']


In [10]:
print(result.logs[2].code)

# -*- coding: utf-8 -*-
"""
Corrected pipeline: no data leakage.
- Load data
- Train‑test split
- Build pipeline (StandardScaler → PolynomialFeatures → VarianceThreshold → SelectKBest)
- Fit only on training data
- Transform both train and test
- Show shapes and simple model performance
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Load dataset
df = pd.read_csv("data.csv")

# Separate features and target
X = df.drop(columns=["target"])
y = df["target"]

# Train‑test split (stratify to keep class balance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Build pipeline components
scaler = StandardScaler()
poly = PolynomialFeatures(deg

In [11]:
%%writefile src/orchestrator.py
from dataclasses import dataclass, field
from pathlib import Path

from src.agents.coder import solve_step
from src.agents.planner import make_plan
from src.agents.reviewer import review_step
from src.profiler import quick_profile


@dataclass
class StepLog:
    """A record of what happened on one plan step, for the final report."""
    step_id: int
    title: str
    task: str
    coder_success: bool
    coder_attempts: int
    approved: bool
    review_reason: str
    leakage_flags: list
    code: str
    stdout: str
    stderr: str


@dataclass
class RunResult:
    """Everything about one full pipeline run, from goal to finish (or failure)."""
    success: bool
    plan: object                 # the Plan object from the planner
    logs: list = field(default_factory=list)
    failed_at_step: int = None   # None if the whole plan completed


def _files_in(workspace: Path) -> list:
    """List files currently in the workspace, so we can tell the coder what already exists."""
    return sorted(p.name for p in workspace.iterdir() if p.is_file())


def run_pipeline(csv_path, goal: str, workspace: Path,
                 max_step_retries: int = 2, coder_attempts_per_try: int = 4,
                 verbose: bool = True) -> RunResult:
    """Run the full agent pipeline: plan the goal, then execute and review each step.

    csv_path: the dataset to work on (will be copied into the workspace as data.csv)
    goal: the plain-English objective, e.g. "predict customer churn"
    workspace: a folder the sandbox will use; created if it doesn't exist
    """
    workspace = Path(workspace)
    workspace.mkdir(parents=True, exist_ok=True)

    # Put the dataset where the sandbox expects it
    import shutil
    shutil.copy(csv_path, workspace / "data.csv")

    profile = quick_profile(workspace / "data.csv")

    if verbose:
        print("[orchestrator] Planning...")
    plan = make_plan(goal, profile)
    if verbose:
        print(f"[orchestrator] Plan has {len(plan.steps)} steps, problem_type={plan.problem_type}")

    logs = []

    for step in plan.steps:
        if verbose:
            print(f"\n[orchestrator] --- Step {step.id}: {step.title} ---")

        # Tell the coder what files already exist from previous steps,
        # so it builds on prior work instead of assuming a blank slate.
        existing_files = [f for f in _files_in(workspace) if f != "step.py"]
        task_with_context = (
            f"{step.task}\n\n"
            f"IMPORTANT: If you save any processed/transformed dataset to CSV, you MUST include "
            f"the target column (keep its original name) alongside the features — later steps "
            f"will need to load labels from these files, not just the feature values.\n\n"
            f"Files already present in the working directory from earlier steps "
            f"(read/reuse them as needed, e.g. via pd.read_csv or joblib.load): {existing_files}"
        )

        review = None
        outcome = None

        for retry in range(max_step_retries + 1):
            outcome = solve_step(task_with_context, profile, workspace,
                                max_attempts=coder_attempts_per_try, verbose=verbose)

            if not outcome.success:
                # The coder itself never got working code; no point reviewing broken code.
                break

            review = review_step(step.task, outcome.code, outcome.result.stdout)
            if review.approved:
                break

            if verbose:
                print(f"[orchestrator] Reviewer rejected (retry {retry + 1}/{max_step_retries}): "
                      f"{review.reason}")
            # Feed the reviewer's specific objection back in as extra task context
            task_with_context = (
                f"{step.task}\n\n"
                f"IMPORTANT: If you save any processed/transformed dataset to CSV, you MUST include "
                f"the target column (keep its original name) alongside the features — later steps "
                f"will need to load labels from these files, not just the feature values.\n\n"
                f"Your previous attempt was rejected by a reviewer for this reason: {review.reason}\n"
                f"Leakage/quality flags: {review.leakage_flags}\n"
                f"Write a corrected version that avoids this problem.\n\n"
                f"Files already present in the working directory: {existing_files}"
            )

        log = StepLog(
            step_id=step.id, title=step.title, task=step.task,
            coder_success=outcome.success, coder_attempts=outcome.attempts,
            approved=bool(review and review.approved),
            review_reason=review.reason if review else "Coder never produced working code.",
            leakage_flags=review.leakage_flags if review else [],
            code=outcome.code, stdout=outcome.result.stdout,
            stderr=outcome.result.stderr,
        )
        logs.append(log)

        if not log.approved:
            if verbose:
                print(f"[orchestrator] Step {step.id} FAILED. Stopping pipeline.")
            return RunResult(success=False, plan=plan, logs=logs, failed_at_step=step.id)

    if verbose:
        print("\n[orchestrator] All steps completed and approved.")
    return RunResult(success=True, plan=plan, logs=logs)

Overwriting src/orchestrator.py


In [12]:
from src.orchestrator import run_pipeline
from src.config import RUNS_DIR

ws2 = RUNS_DIR / "part4_full_run_v2" / "workspace"
ws2.mkdir(parents=True, exist_ok=True)

result2 = run_pipeline(
    csv_path=RUNS_DIR / "part4_full_run" / "source_data.csv",   # reuse the same source CSV
    goal="Predict whether a tumor is malignant or benign, using the 'target' column.",
    workspace=ws2,
)

print("\n\nOverall success:", result2.success)
if not result2.success:
    print("Failed at step:", result2.failed_at_step)

[orchestrator] Planning...
[orchestrator] Plan has 6 steps, problem_type=classification

[orchestrator] --- Step 1: Data Exploration ---
[coder] attempt 1/4: FAILED
[coder] attempt 2/4: OK

[orchestrator] --- Step 2: Missing Values & Encoding ---
[coder] attempt 1/4: OK

[orchestrator] --- Step 3: Feature Engineering/Selection ---
[coder] attempt 1/4: OK
[orchestrator] Reviewer rejected (retry 1/2): The code correctly standardizes, applies PCA, and selects top correlated features as requested, and no leakage is evident in the preprocessing steps.
[coder] attempt 1/4: OK

[orchestrator] --- Step 4: Model Training & Comparison ---
[coder] attempt 1/4: FAILED
[coder] attempt 2/4: OK

[orchestrator] --- Step 5: Best Model Evaluation ---
[coder] attempt 1/4: OK

[orchestrator] --- Step 6: Model Persistence ---
[coder] attempt 1/4: OK

[orchestrator] All steps completed and approved.


Overall success: True


In [13]:
for log in result2.logs:
    print(f"Step {log.step_id} ({log.title}): approved={log.approved}, "
          f"coder_attempts={log.coder_attempts}, reason={log.review_reason}")

Step 1 (Data Exploration): approved=True, coder_attempts=2, reason=The code correctly loads the data, computes summary statistics, visualizes feature distributions, checks class balance, and reports pairwise correlations without any data leakage.
Step 2 (Missing Values & Encoding): approved=True, coder_attempts=1, reason=The script correctly checks and imputes missing values, one‑hot encodes categorical features without touching the target, and preserves the target as an integer.
Step 3 (Feature Engineering/Selection): approved=True, coder_attempts=1, reason=The code correctly fits scalers and PCA on the training set only, computes correlations without using test data, and saves the processed datasets as required.
Step 4 (Model Training & Comparison): approved=True, coder_attempts=2, reason=The code correctly performs a stratified train/test split, uses cross‑validation on the training set only, and evaluates both classifiers on the held‑out test set without leaking target information.

In [14]:
import os
print(sorted(os.listdir(ws2)))

['area_error_dist.png', 'compactness_error_dist.png', 'concave_points_error_dist.png', 'concavity_error_dist.png', 'corr_selected_data.csv', 'correlation_heatmap.png', 'data.csv', 'final_model.joblib', 'fractal_dimension_error_dist.png', 'logreg_model.joblib', 'mean_area_dist.png', 'mean_compactness_dist.png', 'mean_concave_points_dist.png', 'mean_concavity_dist.png', 'mean_fractal_dimension_dist.png', 'mean_perimeter_dist.png', 'mean_radius_dist.png', 'mean_smoothness_dist.png', 'mean_symmetry_dist.png', 'mean_texture_dist.png', 'pca_data.csv', 'perimeter_error_dist.png', 'processed_corr.csv', 'processed_data.csv', 'processed_pca.csv', 'radius_error_dist.png', 'rf_model.joblib', 'roc_curve.png', 'scaled_data.csv', 'smoothness_error_dist.png', 'step.py', 'symmetry_error_dist.png', 'test_split.csv', 'texture_error_dist.png', 'train_split.csv', 'worst_area_dist.png', 'worst_compactness_dist.png', 'worst_concave_points_dist.png', 'worst_concavity_dist.png', 'worst_fractal_dimension_dist.p

In [15]:

import pandas as pd
test_df = pd.read_csv(ws2 / "test_split.csv")   # or whichever processed file Step 5 actually loaded from
print(test_df.columns.tolist())

['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'mean smoothness', 'mean compactness', 'mean concavity', 'mean concave points', 'mean symmetry', 'mean fractal dimension', 'radius error', 'texture error', 'perimeter error', 'area error', 'smoothness error', 'compactness error', 'concavity error', 'concave points error', 'symmetry error', 'fractal dimension error', 'worst radius', 'worst texture', 'worst perimeter', 'worst area', 'worst smoothness', 'worst compactness', 'worst concavity', 'worst concave points', 'worst symmetry', 'worst fractal dimension', 'target']


In [17]:
print(result2.success)


True
